# 05 XGBoost

This notebook analyses the XGBoost benchmark for hourly Dutch day-ahead prices.

What the model does:
- XGBoost fits an ensemble of boosted decision trees
- it can learn non-linear interactions between lagged prices and calendar variables without manually specifying them

Feature sets used now:
- `FS1`: lagged DA prices only
- `FS2`: lagged DA prices plus forecast-known calendar features

Strengths and weaknesses:
- strength: flexible non-linear pattern learning with modest training times
- weakness: lower transparency than a linear model, so later interpretation work should include grouped ablation and SHAP / importance analysis

Leakage prevention:
- the same walk-forward engine is used as for every other model
- the model is refit daily and only sees information available before each `08:00` D-1 origin

In [ ]:
from pathlib import Path
import pandas as pd

run_root = Path('data/02_Forecasting/01_DA_prices/hourly_da/runs')
latest_run = sorted(run_root.glob('*_xgboost_benchmark'))[-1]
latest_run

In [ ]:
metrics_overall = pd.read_csv(latest_run / 'metrics_overall.csv')
metrics_by_lead_day = pd.read_csv(latest_run / 'metrics_by_lead_day.csv')
timing_summary = pd.read_csv(latest_run / 'origin_timing_summary.csv')
dm_results = pd.read_csv(latest_run / 'diebold_mariano_results.csv')

display(metrics_overall)
display(metrics_by_lead_day[metrics_by_lead_day['model'].str.startswith('xgboost_')])
display(timing_summary[timing_summary['model'].str.startswith('xgboost_')])

Tuning note:

- only a controlled first configuration is implemented here
- later refinement can tune tree depth, number of estimators, learning rate, and regularization on the shared validation framework
- later feature-family expansion should still proceed group by group rather than adding all exogenous candidates at once

In [ ]:
dm_results[dm_results['challenger_model'].str.startswith('xgboost_')]